# Installed Capacity by Technology
Stacked bar charts of `TotalCapacityAnnual` (GW) per country and scenario.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SCENARIOS = {
    'REF' : 'results/NamiraN_MSc_REF',
    # 'NECP': 'results/NamiraN_MSc_NECP',   # uncomment when ready
}

# All years 2015–2050
PLOT_YEARS = list(range(2015, 2051))

# 15 modelled countries
COUNTRIES = ['AT','BE','DE','DK','EE','FI','FR','LT','LU','LV','NL','NO','PL','SE','UK']

COUNTRY_NAMES = {
    'AT':'Austria',     'BE':'Belgium',     'DE':'Germany',   'DK':'Denmark',
    'EE':'Estonia',     'FI':'Finland',     'FR':'France',    'LT':'Lithuania',
    'LU':'Luxembourg',  'LV':'Latvia',      'NL':'Netherlands','NO':'Norway',
    'PL':'Poland',      'SE':'Sweden',      'UK':'United Kingdom'
}

# Fuel codes (chars 2-3 of technology name)
FUEL_MAP = {
    'BF': 'BF',  # Biofuel
    'BM': 'BM',  # Biomass
    'CO': 'CO',  # Coal
    'GO': 'GO',  # Geothermal
    'HF': 'HF',  # Heavy fuel / Oil
    'HY': 'HY',  # Hydro
    'NG': 'NG',  # Natural gas
    'NU': 'NU',  # Nuclear
    'OC': 'OC',  # Ocean
    'SO': 'SO',  # Solar
    'WI': 'WI',  # Wind
    'WS': 'WS',  # Waste
}

# Colours matching reference figure style
FUEL_COLORS = {
    'BF': '#9DC3E6',
    'BM': '#4472C4',
    'CO': '#FF0000',
    'GO': '#70AD47',
    'HF': '#A9D18E',
    'HY': '#7030A0',
    'NG': '#00B0F0',
    'NU': '#FFC000',
    'OC': '#002060',
    'SO': '#203864',
    'WI': '#843C00',
    'WS': '#375623',
}

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SCENARIOS = {
    'REF' : 'results/NamiraN_MSc_REF',
    # 'NECP': 'results/NamiraN_MSc_NECP',   # uncomment when ready
}

PLOT_YEARS = [2015, 2020, 2025, 2030, 2035, 2040, 2045, 2050]

COUNTRIES = [
    'AT','BE','CH','CZ','DE','DK','EE','FI',
    'FR','HU','IT','LT','LU','LV','NL','NO',
    'PL','SE','SI','UK'
]

COUNTRY_NAMES = {
    'AT':'Austria',     'BE':'Belgium',      'CH':'Switzerland',  'CZ':'Czech Republic',
    'DE':'Germany',     'DK':'Denmark',      'EE':'Estonia',      'FI':'Finland',
    'FR':'France',      'HU':'Hungary',      'IT':'Italy',        'LT':'Lithuania',
    'LU':'Luxembourg',  'LV':'Latvia',       'NL':'Netherlands',  'NO':'Norway',
    'PL':'Poland',      'SE':'Sweden',       'SI':'Slovenia',     'UK':'United Kingdom'
}

# Fuel codes (chars 2-3 of technology name) and display labels
FUEL_MAP = {
    'BF': 'BF',  # Biofuel
    'BM': 'BM',  # Biomass
    'CO': 'CO',  # Coal
    'GO': 'GO',  # Geothermal
    'HF': 'HF',  # Heavy fuel / Oil
    'HY': 'HY',  # Hydro
    'NG': 'NG',  # Natural gas
    'NU': 'NU',  # Nuclear
    'OC': 'OC',  # Ocean
    'SO': 'SO',  # Solar
    'WI': 'WI',  # Wind
    'WS': 'WS',  # Waste
}

# Colours matching reference figure style
FUEL_COLORS = {
    'BF': '#9DC3E6',  # light blue
    'BM': '#4472C4',  # blue
    'CO': '#FF0000',  # red
    'GO': '#70AD47',  # green
    'HF': '#A9D18E',  # light green
    'HY': '#7030A0',  # purple
    'NG': '#00B0F0',  # cyan
    'NU': '#FFC000',  # orange
    'OC': '#002060',  # dark navy
    'SO': '#203864',  # dark blue
    'WI': '#843C00',  # dark brown
    'WS': '#375623',  # dark green
}

In [ ]:
def plot_capacity(df, countries, years, scenarios, save_path=None, ncols=3):
    """Plot stacked bar charts of installed capacity for each country."""
    nrows       = int(np.ceil(len(countries) / ncols))
    fig, axes   = plt.subplots(nrows, ncols,
                               figsize=(6 * ncols, 4 * nrows),
                               sharey=False)
    axes        = axes.flatten()
    scen_labels = list(scenarios.keys())
    bar_width   = 0.8 / len(scen_labels)
    x           = np.arange(len(years))
    fuels       = list(FUEL_MAP.keys())

    # X-axis: label every 3 years starting from the first year (2015, 2018, 2021, ...)
    tick_positions = [i for i, y in enumerate(years) if (y - years[0]) % 3 == 0]
    tick_labels    = [str(years[i]) for i in tick_positions]

    for ax_idx, country in enumerate(countries):
        ax    = axes[ax_idx]
        pivot = build_capacity_df(df, country, years)

        for s_idx, scen in enumerate(scen_labels):
            if scen not in pivot.index.get_level_values('scenario'):
                continue
            data   = pivot.xs(scen, level='scenario').reindex(years, fill_value=0)
            offset = (s_idx - (len(scen_labels) - 1) / 2) * bar_width
            bottom = np.zeros(len(years))

            for fuel in fuels:
                vals = data[fuel].values
                if vals.sum() < 1e-6:
                    continue
                label = FUEL_MAP[fuel] if s_idx == 0 else '_nolegend_'
                ax.bar(x + offset, vals, bar_width,
                       bottom=bottom,
                       color=FUEL_COLORS[fuel],
                       label=label,
                       edgecolor='none')
                bottom += vals

        ax.set_title(COUNTRY_NAMES.get(country, country), fontsize=10, fontweight='bold')
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=45, fontsize=7)
        ax.set_ylabel('GW', fontsize=8)
        ax.yaxis.set_tick_params(labelsize=7)
        ax.spines[['top', 'right']].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        ax.set_axisbelow(True)
        ax.set_xlim(-0.5, len(years) - 0.5)

    # Hide unused subplots
    for ax_idx in range(len(countries), len(axes)):
        axes[ax_idx].set_visible(False)

    # Shared legend at the bottom
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center',
               ncol=len(fuels), fontsize=9,
               frameon=False, bbox_to_anchor=(0.5, -0.02))

    scen_str = ' vs '.join(scen_labels)
    fig.suptitle(f'Installed Capacity (GW) — {scen_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {save_path}')
    plt.show()


plot_capacity(
    cap_df, COUNTRIES, PLOT_YEARS, SCENARIOS,
    save_path='results/capacity_all_countries.png'
)

In [ ]:
def plot_capacity(df, countries, years, scenarios, save_path=None, ncols=4):
    """Plot stacked bar charts of installed capacity for each country."""
    nrows = int(np.ceil(len(countries) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows),
                             sharey=False)
    axes = axes.flatten()

    scen_labels = list(scenarios.keys())
    bar_width   = 0.7 / len(scen_labels)
    x           = np.arange(len(years))
    fuels       = list(FUEL_MAP.keys())

    for ax_idx, country in enumerate(countries):
        ax = axes[ax_idx]
        pivot = build_capacity_df(df, country, years)

        for s_idx, scen in enumerate(scen_labels):
            if scen not in pivot.index.get_level_values('scenario'):
                continue
            data = pivot.xs(scen, level='scenario').reindex(years, fill_value=0)
            offset = (s_idx - (len(scen_labels) - 1) / 2) * bar_width
            bottom = np.zeros(len(years))

            for fuel in fuels:
                vals = data[fuel].values
                if vals.sum() < 1e-6:
                    continue
                label = FUEL_MAP[fuel] if s_idx == 0 else '_nolegend_'
                ax.bar(x + offset, vals, bar_width,
                       bottom=bottom,
                       color=FUEL_COLORS[fuel],
                       label=label,
                       edgecolor='none')
                bottom += vals

        ax.set_title(COUNTRY_NAMES.get(country, country), fontsize=10, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(years, rotation=45, fontsize=7)
        ax.set_ylabel('GW', fontsize=8)
        ax.yaxis.set_tick_params(labelsize=7)
        ax.spines[['top', 'right']].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        ax.set_axisbelow(True)

    # Hide unused subplots
    for ax_idx in range(len(countries), len(axes)):
        axes[ax_idx].set_visible(False)

    # Legend from first subplot
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center',
               ncol=len(fuels), fontsize=8,
               frameon=False, bbox_to_anchor=(0.5, -0.02))

    scen_str = ' vs '.join(scen_labels)
    fig.suptitle(f'Installed Capacity (GW) — {scen_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {save_path}')
    plt.show()


plot_capacity(
    cap_df, COUNTRIES, PLOT_YEARS, SCENARIOS,
    save_path='results/capacity_all_countries.png'
)

In [ ]:
# ── Single country detailed view ──────────────────────────────────────────────
def plot_single_country(df, country, years, scenarios):
    fig, ax = plt.subplots(figsize=(10, 5))
    scen_labels = list(scenarios.keys())
    bar_width   = 0.7 / len(scen_labels)
    x           = np.arange(len(years))
    fuels       = list(FUEL_MAP.keys())
    pivot       = build_capacity_df(df, country, years)

    for s_idx, scen in enumerate(scen_labels):
        if scen not in pivot.index.get_level_values('scenario'):
            continue
        data   = pivot.xs(scen, level='scenario').reindex(years, fill_value=0)
        offset = (s_idx - (len(scen_labels) - 1) / 2) * bar_width
        bottom = np.zeros(len(years))
        for fuel in fuels:
            vals = data[fuel].values
            if vals.sum() < 1e-6:
                continue
            label = f'{FUEL_MAP[fuel]} ({scen})' if len(scen_labels) > 1 else FUEL_MAP[fuel]
            ax.bar(x + offset, vals, bar_width, bottom=bottom,
                   color=FUEL_COLORS[fuel], label=label, edgecolor='none')
            bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(years, rotation=45)
    ax.set_ylabel('GW')
    ax.set_title(f'Installed Capacity — {COUNTRY_NAMES.get(country, country)}',
                 fontsize=13, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    ax.legend(loc='upper left', fontsize=8, frameon=False, ncol=2)
    plt.tight_layout()
    plt.show()


# Change country code to inspect any country
plot_single_country(cap_df, 'PL', PLOT_YEARS, SCENARIOS)